In [203]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import mlflow
import xgboost as xgb
from hyperopt import tpe, hp, STATUS_OK, fmin, Trials
from hyperopt.pyll import scope

In [204]:
mlflow.set_experiment('road_risk')

<Experiment: artifact_location='file:///Users/aravindrajeshmenon/Documents/DataScienceProjects/Projects/road_accident_risk/road_risk/notebooks/mlruns/549507082084492537', creation_time=1760406937748, experiment_id='549507082084492537', last_update_time=1760406937748, lifecycle_stage='active', name='road_risk', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [205]:
mlflow.get_tracking_uri()

'file:///Users/aravindrajeshmenon/Documents/DataScienceProjects/Projects/road_accident_risk/road_risk/notebooks/mlruns'

In [206]:
df = pd.read_csv('../data/train.csv')
x_test_df = pd.read_csv('../data/test.csv')

In [207]:
df=  df.drop(['id'], axis = 1)
x_test = x_test_df.drop(['id', 'school_season', 'road_signs_present'], axis = 1)

In [208]:
df.isna().sum()

road_type                 0
num_lanes                 0
curvature                 0
speed_limit               0
lighting                  0
weather                   0
road_signs_present        0
public_road               0
time_of_day               0
holiday                   0
school_season             0
num_reported_accidents    0
accident_risk             0
dtype: int64

In [209]:
df.sample()

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
48904,highway,2,0.28,35,night,foggy,False,True,evening,True,True,0,0.48


In [210]:
df.columns

Index(['road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting',
       'weather', 'road_signs_present', 'public_road', 'time_of_day',
       'holiday', 'school_season', 'num_reported_accidents', 'accident_risk'],
      dtype='object')

In [211]:
x = df.drop(['accident_risk', 'school_season', 'road_signs_present'], axis = 1)
y = df['accident_risk'].values

In [212]:
x_train, x_val, y_train, y_val= train_test_split(x,y,train_size = 0.8, random_state = 42)


In [213]:
num_cols = x.select_dtypes(['int', 'float']).columns.to_list()
cat_cols = x.select_dtypes(['object', 'bool']).columns.to_list()

print(num_cols)
print(cat_cols)

['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
['road_type', 'lighting', 'weather', 'public_road', 'time_of_day', 'holiday']


In [214]:
dv = DictVectorizer(sparse = False)

train_dict = x_train[cat_cols + num_cols].to_dict(orient = 'records')
x_train_encoded = dv.fit_transform(train_dict)

val_dict = x_val[cat_cols + num_cols].to_dict(orient = 'records')
x_val_encoded = dv.transform(val_dict)


test_dict = x_test[cat_cols + num_cols].to_dict(orient = 'records')
x_test_encoded = dv.transform(test_dict)

In [215]:
with mlflow.start_run():
    mlflow.set_tag("model", "linear_regression")
    lr = LinearRegression()
    lr.fit(x_train_encoded, y_train)
    y_pred = lr.predict(x_val_encoded)
    rmse = root_mean_squared_error(y_val, y_pred)
    print(rmse)
    mlflow.log_metric("rmse", rmse)

0.07353038492000742


In [216]:
with mlflow.start_run():
    mlflow.set_tag("model", "Lasso")
    lasso = Lasso(alpha = 1)
    lasso.fit(x_train_encoded, y_train)
    y_pred = lasso.predict(x_val_encoded)
    rmse = root_mean_squared_error(y_val, y_pred)
    print(rmse)
    mlflow.log_metric("rmse", rmse)

0.16271187689427052


In [217]:
with mlflow.start_run():
    mlflow.set_tag("model", "Ridge")
    ridge = Ridge(alpha = 0.001)
    ridge.fit(x_train_encoded, y_train)
    y_pred = ridge.predict(x_val_encoded)
    rmse = root_mean_squared_error(y_val, y_pred)
    print(rmse)
    mlflow.log_metric("rmse", rmse)
    

0.0735303849125793


In [218]:
train = xgb.DMatrix(x_train_encoded, label = y_train)
valid = xgb.DMatrix(x_val_encoded, y_val)
test = xgb.DMatrix(x_test_encoded)

In [219]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", 'xgboost')
        mlflow.log_params(params)
        booster = xgb.train(
            params = params, 
            dtrain = train, 
            num_boost_round = 200, 
            evals=[(train, "train"), (valid, "validation")],
            early_stopping_rounds = 50,
            verbose_eval = False
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        mlflow.xgboost.log_model(booster, name = "xgb_mlflow")
        return {'loss': rmse, 'status': STATUS_OK}
        

In [220]:
search_space = {
    'max_depth' : scope.int(hp.quniform("max_depth", 10,30,1)),
    'learning_rate' : hp.loguniform("learning_rate", -3, 0),
    'min_child_weight' : hp.choice("min_samples_split", [2,5,7,10]),
    'subsample' : hp.uniform("subsample", 0.7,1),
    'colsample_bytree' : hp.uniform("colsample_bytree", 0.7,1),
    'gamma' : hp.uniform("gamma", 0, 0.5),
    'reg_alpha' : hp.loguniform("reg_alpha", -5, 0),
    "reg_lambda" : hp.loguniform("reg_lambda", -5,0),
    'objective' : 'reg:squarederror'

}

In [221]:
trials = Trials()
best_result = fmin(
    fn = objective, 
    space = search_space, 
    algo = tpe.suggest, 
    max_evals = 50
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/mlflow/xgboost/__init__.py:169: UserWarning: [13:00:35] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)

2025/10/28 13:00:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



  2%|▏         | 1/50 [00:05<04:52,  5.98s/trial, best loss: 0.05639189872986167]

/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/mlflow/xgboost/__init__.py:169: UserWarning: [13:00:39] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)

2025/10/28 13:00:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



  4%|▍         | 2/50 [00:09<03:37,  4.52s/trial, best loss: 0.05639189872986167]

...

 96%|█████████▌| 48/50 [03:25<00:11,  5.94s/trial, best loss: 0.0562074424602371]

/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/mlflow/xgboost/__init__.py:169: UserWarning: [13:03:58] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)

2025/10/28 13:04:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



 98%|█████████▊| 49/50 [03:28<00:05,  5.03s/trial, best loss: 0.0562074424602371]

/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/mlflow/xgboost/__init__.py:169: UserWarning: [13:04:03] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)

2025/10/28 13:04:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



100%|██████████| 50/50 [03:34<00:00,  4.28s/trial, best loss: 0.0562074424602371]


In [222]:
best_params = {
    'learning_rate': 0.062148724176815065,
    'objective': 'reg:squarederror',
    'colsample_bytree': 0.9897467268146862,
    'reg_alpha': 0.04303072152137027,
    'subsample': 0.9686545325132151,
    'reg_lambda': 0.011873421662215236,
    'gamma': 0.019715130360417568,
    'max_depth': 19,
    'min_child_weight': 7
}

In [223]:
with mlflow.start_run():
        mlflow.set_tag("model", 'xgboost')
        mlflow.log_params(best_params)
        booster = xgb.train(
            params = best_params, 
            dtrain = train, 
            num_boost_round = 200, 
            evals=[(train, "train"), (valid, "validation")],
            early_stopping_rounds = 50,
            verbose_eval = False
        )
        y_pred_test = booster.predict(test)
        mlflow.xgboost.log_model(booster, name = "xgb_mlflow")

/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/mlflow/xgboost/__init__.py:169: UserWarning: [13:04:10] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)
2025/10/28 13:04:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [224]:
y_pred_df = pd.DataFrame(data = {'id': x_test_df['id'].values, 'accident_risk' : y_pred_test})

In [225]:
y_pred_df

,id,accident_risk
0,517754,0.306618
1,517755,0.123340
2,517756,0.184344
3,517757,0.323602
4,517758,0.407911
...,...,...
172580,690334,0.107272
172581,690335,0.524965
172582,690336,0.250725
172583,690337,0.128537


In [226]:
y_pred_df.to_csv('../data/sample_submissions.csv', index = False)

In [227]:
with mlflow.start_run():
    mlflow.set_tag("model", 'stacking_ensemble')
    
    # Define base models
    base_models = [
        ('xgb', XGBRegressor(**best_params, n_estimators=500, eval_metric='rmse')),
        ('rf', RandomForestRegressor(n_estimators=150, max_depth=15, random_state=42)),
        ('lgbm', LGBMRegressor(n_estimators=1000, max_depth=15, random_state=42))
    ]
    
    meta_model = Ridge(alpha = 0.01)
    
    # Create stacking regressor
    stacking_model = StackingRegressor(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5, 
        verbose=2 
    )
    
    # Train
    stacking_model.fit(x_train_encoded, y_train)
    
    y_pred_val = stacking_model.predict(x_val_encoded)
    rmse = root_mean_squared_error(y_val, y_pred_val)

    print(rmse)
    
    # Log model
    mlflow.sklearn.log_model(stacking_model, "stacking_model")
    mlflow.log_metric("rmse", rmse)
    
    # Log params
    mlflow.log_param("base_models", "xgb+rf+lgbm")
    mlflow.log_param("meta_model", "gradient_boosting")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003054 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 175
[LightGBM] [Info] Number of data points in the train set: 414203, number of used features: 18
[LightGBM] [Info] Start training from score 0.352605


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    7.1s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  8.8min finished


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002529 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 175
[LightGBM] [Info] Number of data points in the train set: 331362, number of used features: 18
[LightGBM] [Info] Start training from score 0.352332


/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002207 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 176
[LightGBM] [Info] Number of data points in the train set: 331362, number of used features: 18
[LightGBM] [Info] Start training from score 0.352555


...

/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002066 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 177
[LightGBM] [Info] Number of data points in the train set: 331363, number of used features: 18
[LightGBM] [Info] Start training from score 0.352721


/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   27.5s finished
/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2025/10/28 13:16:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


0.05615805319471712


2025/10/28 13:16:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [228]:
y_test_stack_pred = stacking_model.predict(x_test_encoded)

y_pred_stack_df = pd.DataFrame(data = {'id': x_test_df['id'].values, 'accident_risk' : y_test_stack_pred})

y_pred_stack_df.to_csv('../data/sample_submission.csv', index = False)

/opt/anaconda3/envs/churnenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
